# 16 — Server GRPO object training: profile → env → logs → RLCluster → learner

Эта тетрадка — серверный runbook без запуска `scripts/*.py`. Все ключевые объекты создаются прямо в ячейках: profile, config, topology, workload, logger, validation evidence, CrafText agentic env, Qwen assets, `RLCluster`, `GRPOLearner` и training iterator.

Цель: на машине с GPU пройти по ячейкам и увидеть, что именно инстанцируется, куда пишутся логи, где появляются validation trajectories и куда Tunix должен складывать checkpoints.


## 0. External preparation

Перед real run окружение и веса должны быть готовы локально:

```bash
pyenv exec python -m uv sync --all-extras
pyenv exec python -m uv pip install -U "jax[cuda13]"  # или jax[cuda12] под драйвер сервера
```

Notebook ничего не скачивает неявно. Snapshot берётся из profile: `artifacts/models/qwen25-05b-instruct`.


In [ ]:
from __future__ import annotations

import json
from collections.abc import Iterator
from pathlib import Path

import jax
import numpy as np

from tunix_craftext.artifacts.observability import (
    JsonlRunLogger,
    MetricRecord,
    RunArtifact,
    ValidationTrajectoryRecord,
    checkpoint_artifact,
    read_jsonl,
    validation_trajectory_artifact,
)
from tunix_craftext.env.agentic_craftext import (
    CrafTextAgenticEnvironment,
    CrafTextStepTool,
    agentic_environment_kwargs,
)
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.training.agentic_grpo_smoke import (
    collect_scripted_grpo_group_sync,
    save_scripted_grpo_smoke,
)
from tunix_craftext.training.grpo_profile import (
    build_grpo_evidence_manifest,
    load_agentic_grpo_profile,
)
from tunix_craftext.tunix import (
    build_agentic_grpo_cluster,
    load_agentic_grpo_qwen_assets,
    load_tunix_topology,
    pinned_qwen_tensor_shape,
    role_to_meshes,
    validate_agentic_grpo_preflight,
)
from tunix_craftext.models.tunix_adapter import load_qwen_hf_tokenizer


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

PROFILE_PATH = ROOT / 'configs/grpo/qwen_agentic_local.yaml'
print('repo:', ROOT)
print('jax backend:', jax.default_backend())
print('jax devices:')
for device in jax.devices():
    print(' ', device)


## 1. Profile, config, runtime and local logger

Это базовая связка одного training run: YAML profile задаёт workload/evidence/model, MVP config задаёт CrafText/CagedCrafText runtime, `JsonlRunLogger` пишет локальную truth-source телеметрию.


In [ ]:
profile = load_agentic_grpo_profile(PROFILE_PATH)
config_path = ROOT / profile.environment_config
topology_path = ROOT / profile.topology_config
snapshot = ROOT / profile.model.snapshot
run_dir = ROOT / profile.evidence.root

config = load_mvp_config(config_path)
runtime = build_craftext_runtime(config)
logger = JsonlRunLogger(run_dir)

for path in (
    run_dir,
    ROOT / profile.evidence.checkpoints,
    run_dir / 'validation',
    run_dir / 'profiles',
):
    path.mkdir(parents=True, exist_ok=True)

print('run:', profile.run.name)
print('goal:', profile.run.goal)
print('config:', config_path, config_path.is_file())
print('snapshot:', snapshot, snapshot.is_dir())
print('run dir:', run_dir)
print('action count:', runtime.action_count)
print('first actions:', runtime.actions.labels[:8])


## 2. Manual CrafText agentic environment

Тот же environment class потом передаётся в Tunix `GRPOLearner`. Здесь мы поднимаем один экземпляр руками, чтобы увидеть prompt и проверить instruction/world state до загрузки весов.


In [ ]:
env_kwargs = agentic_environment_kwargs(config_path)
manual_task = {
    'goal': profile.run.goal,
    'seed': config.run.seed,
    'horizon': min(config.environment.horizon, 4),
}
manual_env = CrafTextAgenticEnvironment(manual_task, **env_kwargs, group_id=0, pair_index=0)
initial_observation, initial_info = manual_env.reset()

print('env kwargs:', env_kwargs)
print('reset info:', initial_info)
print(initial_observation['question'][:1600])


## 3. Topology/workload preflight before HBM allocation

Эта ячейка не грузит модель. Она проверяет Tunix topology, static workload и mesh divisibility. На сервере она должна пройти до любой тяжёлой ячейки.


In [ ]:
topology = load_tunix_topology(topology_path)
spec = profile.workload
validate_agentic_grpo_preflight(topology, spec, pinned_qwen_tensor_shape())
meshes = role_to_meshes(topology)

print('topology:', topology.name)
print('role devices:', topology.role_to_device_indices)
for role, mesh in meshes.items():
    print(role, mesh.shape, mesh.axis_names)
print('workload:', spec)


## 4. Provenance, metrics and validation evidence before model loading

Сначала пишем локальный evidence. Потом Comet/любой team logger может зеркалить те же records, но JSONL остаётся source of truth.


In [ ]:
manifest = build_grpo_evidence_manifest(profile, profile_path=PROFILE_PATH, repo_root=ROOT)
provenance_path = ROOT / profile.evidence.provenance
provenance_path.parent.mkdir(parents=True, exist_ok=True)
provenance_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')

logger.log_artifact(RunArtifact(profile.run.name, str(PROFILE_PATH), 'config', step=0))
logger.log_artifact(RunArtifact(profile.run.name, str(provenance_path), 'profile', step=0))
logger.log_metric(
    MetricRecord(
        run_id=profile.run.name,
        step=0,
        split='eval',
        phase='notebook_preflight',
        metrics={
            'device_count': len(jax.devices()),
            'snapshot_exists': snapshot.is_dir(),
            'action_count': runtime.action_count,
        },
    )
)
print('provenance:', provenance_path)
print('metrics:', logger.metrics_path)
print('artifacts:', logger.artifacts_path)


## 5. Validation trajectory smoke without LLM weights

Это не training script: мы напрямую вызываем модульный collector для короткой scripted validation group. Так проверяется agentic tool-loop и запись validation trajectory до дорогой загрузки модели.


In [ ]:
scripted_horizon = min(config.environment.horizon, 2)
action_sequences = (
    ('NOOP',) * scripted_horizon,
    ('LEFT',) * scripted_horizon,
)
scripted_results = collect_scripted_grpo_group_sync(
    config_path=config_path,
    goal=profile.run.goal,
    seed=config.run.seed,
    group_id=0,
    action_sequences=action_sequences,
    horizon=scripted_horizon,
)
validation_path = run_dir / 'validation' / 'notebook-scripted-grpo-validation.json'
save_scripted_grpo_smoke(validation_path, scripted_results)
first = scripted_results[0]

logger.log_validation_trajectory(
    ValidationTrajectoryRecord(
        run_id=profile.run.name,
        step=0,
        task_id='notebook-scripted-grpo',
        trajectory_path=str(validation_path),
        return_sum=first.total_reward,
        episode_length=max(len(first.rewards), 1),
        success=True,
        policy_version=0,
        metrics={'mode': 'scripted', 'num_generations': len(scripted_results)},
    )
)
logger.log_artifact(
    validation_trajectory_artifact(
        profile.run.name,
        validation_path,
        step=0,
        task_id='notebook-scripted-grpo',
        policy_version=0,
    )
)
print('validation path:', validation_path)
for result in scripted_results:
    print(result)


## 6. Hard server gate before loading weights

Если эта ячейка падает, training запускать нельзя: либо JAX не видит GPU, либо нет локального snapshot. На локальном CPU её ожидаемо не надо обходить случайно — лучше остановиться.


In [ ]:
if jax.default_backend() == 'cpu':
    raise RuntimeError('JAX backend is CPU. On the server install CUDA JAX and rerun before model allocation.')
if not snapshot.is_dir():
    raise FileNotFoundError(f'Local model snapshot is missing: {snapshot}')

print('server gate passed')
print('backend:', jax.default_backend())
print('snapshot:', snapshot)
print('checkpoint root:', ROOT / profile.evidence.checkpoints)


## 7. Load Qwen actor/reference assets and tokenizer

Первая тяжёлая ячейка. Actor грузится как trainable copy, reference — frozen copy. Tokenizer держим явно, потому что он нужен `QwenChatTemplateParser`.


In [ ]:
assets = load_agentic_grpo_qwen_assets(snapshot, topology)
hf_tokenizer = load_qwen_hf_tokenizer(snapshot)

print('actor:', type(assets.actor))
print('reference:', type(assets.reference))
print('tunix tokenizer:', type(assets.tokenizer))
print('hf tokenizer:', type(hf_tokenizer))


## 8. Build RLCluster and GRPOLearner manually

Здесь нет скрипта-обёртки: cluster и learner создаются прямо из объектов выше.


In [ ]:
from tunix.rl.agentic.agentic_grpo_learner import GRPOConfig, GRPOLearner
from tunix.rl.agentic.agents.tool_agent import ToolAgent
from tunix.rl.agentic.parser.chat_template_parser.parser import QwenChatTemplateParser

cluster = build_agentic_grpo_cluster(topology, spec, assets)
learner = GRPOLearner(
    rl_cluster=cluster,
    algo_config=GRPOConfig(
        num_generations=profile.workload.num_generations,
        max_response_length=profile.workload.max_new_tokens,
        max_concurrency=profile.workload.max_concurrency,
        system_prompt='Use craftext_step for every environment action.',
    ),
    chat_parser=QwenChatTemplateParser(hf_tokenizer, enable_thinking=False),
    agent_class=ToolAgent,
    agent_kwargs={
        'system_prompt': 'Use craftext_step for every environment action.',
        'tool_parser_name': 'qwen',
        'tool_map': {'craftext_step': CrafTextStepTool},
    },
    env_class=CrafTextAgenticEnvironment,
    env_kwargs=env_kwargs,
)

print('cluster:', type(cluster))
print('learner:', type(learner))
print('cluster checkpoint root:', cluster.cluster_config.training_config.checkpoint_root_directory)


## 9. Training batches and direct learner.train

Эта ячейка реально запускает training update. После неё смотрим JSONL evidence и checkpoint root.


In [ ]:
def task_batches(
    *,
    goal: str,
    seed: int,
    batch_size: int,
    count: int,
    horizon: int,
) -> Iterator[dict[str, object]]:
    for batch_index in range(count):
        start = seed + batch_index * batch_size
        yield {
            'goal': [goal] * batch_size,
            'seed': np.arange(start, start + batch_size, dtype=np.int32),
            'horizon': np.full(batch_size, horizon, dtype=np.int32),
        }

train_iterator = task_batches(
    goal=profile.run.goal,
    seed=config.run.seed,
    batch_size=profile.workload.mini_batch_size,
    count=profile.workload.max_steps,
    horizon=config.environment.horizon,
)

try:
    learner.train(train_iterator, skip_jit=False)
finally:
    cluster.close()

logger.log_metric(
    MetricRecord(
        run_id=profile.run.name,
        step=profile.workload.max_steps,
        split='train',
        phase='notebook_train_call_finished',
        metrics={'max_steps': profile.workload.max_steps},
        checkpoint_path=str(ROOT / profile.evidence.checkpoints),
    )
)
logger.log_artifact(
    checkpoint_artifact(
        profile.run.name,
        ROOT / profile.evidence.checkpoints,
        step=profile.workload.max_steps,
        role='tunix-rlcluster',
    )
)
print('training call finished')


## 10. Inspect evidence after the run

Если training прошёл, эти JSONL должны быть первыми файлами, которые мы смотрим перед графиками/Comet: локальные records не зависят от сети.


In [ ]:
print('metrics path:', logger.metrics_path)
print('validation trajectories path:', logger.validation_trajectories_path)
print('artifacts path:', logger.artifacts_path)

print('\nlast metrics:')
for row in read_jsonl(logger.metrics_path)[-5:]:
    print(json.dumps(row, indent=2, ensure_ascii=False, sort_keys=True))

print('\nvalidation trajectory records:')
for row in read_jsonl(logger.validation_trajectories_path)[-5:]:
    print(json.dumps(row, indent=2, ensure_ascii=False, sort_keys=True))

print('\nartifact records:')
for row in read_jsonl(logger.artifacts_path)[-8:]:
    print(json.dumps(row, indent=2, ensure_ascii=False, sort_keys=True))
